In [ ]:
# ============================================================================
# CELL 1: Install packages
# ============================================================================
%pip install qiskit qiskit-aer qiskit-ibm-runtime

In [ ]:
# ============================================================================
# CELL 2: Imports
# ============================================================================
import numpy as np
from typing import List, Tuple, Dict, Optional

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

import matplotlib.pyplot as plt
import time

In [ ]:
# ============================================================================
# CELL 3: Entangled Pairs & Noise Model
# ============================================================================
class EntangledPair:
    """Entangled qubit pair - creates ONLY valid quantum Bell states."""
    def __init__(self, pair_id: int, bell_state: str = 'phi_plus'):
        self.pair_id = pair_id
        # Only allow valid Bell states - reject invalid ones
        valid_states = ['phi_plus', 'phi_minus', 'psi_plus', 'psi_minus']
        if bell_state not in valid_states:
            bell_state = 'phi_plus'
        self.bell_state = bell_state

    def create_bell_state(self):
        """Create valid Bell state - no artificial corruption."""
        qc = QuantumCircuit(2)
        if self.bell_state == 'phi_plus':
            # |Φ+⟩ = (|00⟩ + |11⟩)/√2
            qc.h(0)
            qc.cx(0, 1)
        elif self.bell_state == 'phi_minus':
            # |Φ-⟩ = (|00⟩ - |11⟩)/√2
            qc.x(0)
            qc.h(0)
            qc.cx(0, 1)
        elif self.bell_state == 'psi_plus':
            # |Ψ+⟩ = (|01⟩ + |10⟩)/√2
            qc.h(0)
            qc.cx(0, 1)
            qc.x(1)
        elif self.bell_state == 'psi_minus':
            # |Ψ-⟩ = (|01⟩ - |10⟩)/√2
            qc.x(0)
            qc.h(0)
            qc.cx(0, 1)
            qc.x(1)
        return qc

def generate_entangled_pairs(n_pairs: int, bell_state: str = 'phi_plus') -> List[EntangledPair]:
    return [EntangledPair(i, bell_state) for i in range(n_pairs)]


def create_noise_model(gate_error_1q: float = 0.001,
                       gate_error_2q: float = 0.01,
                       readout_error: float = 0.01) -> NoiseModel:
    """
    Create realistic noise model with proper qubit matching.

    Typical quantum hardware error rates:
    - 1-qubit gates: 0.1% (0.001)
    - 2-qubit gates: 1% (0.01) - 10x worse than 1-qubit
    - Readout: 1-2% (0.01-0.02)
    """
    noise_model = NoiseModel()

    # 1-qubit gate errors
    if gate_error_1q > 0:
        error_1q = depolarizing_error(gate_error_1q, 1)
        noise_model.add_all_qubit_quantum_error(error_1q, ['h', 'x', 'z', 'y', 'ry'])

    # 2-qubit gate errors - Use 2-qubit error for CX
    if gate_error_2q > 0:
        error_2q = depolarizing_error(gate_error_2q, 2)
        noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])

    # Readout errors -  Use proper ReadoutError
    if readout_error > 0:
        readout_probs = [[1 - readout_error, readout_error],
                         [readout_error, 1 - readout_error]]
        error_readout = ReadoutError(readout_probs)
        noise_model.add_all_qubit_readout_error(error_readout)

    return noise_model

class QuantumMeasurement:
    """Quantum measurement with noise."""
    def __init__(self, noise_model: Optional[NoiseModel] = None):
        self.simulator = AerSimulator(noise_model=noise_model) if noise_model else AerSimulator()

    def measure_entangled_pair(self, pair: EntangledPair, alice_angle: float, bob_angle: float) -> Tuple[int, int]:
        qc = pair.create_bell_state()
        qc.ry(2 * alice_angle, 0)
        qc.ry(2 * bob_angle, 1)
        qc.measure_all()

        result = self.simulator.run(transpile(qc, self.simulator), shots=1).result()
        bitstring = list(result.get_counts().keys())[0]

        alice_result = 2 * int(bitstring[1]) - 1
        bob_result = 2 * int(bitstring[0]) - 1
        return alice_result, bob_result

In [ ]:
# ============================================================================
# CELL 4: Basis Exchange
# ============================================================================
def exchange_basis_information(alice_bases: List[int], bob_bases: List[int]) -> Tuple[List[int], List[int]]:
    """
    E91 Protocol basis matching:
    - Alice uses angles: [0°, 45°, 90°] = bases [0, 1, 2]
    - Bob uses angles: [45°, 90°, 135°] = bases [0, 1, 2]

    For KEY GENERATION, we need specific angle pairs that give perfect correlation:
    - Alice 45° (basis 1) with Bob 45° (basis 0) → should be perfectly correlated
    - Alice 90° (basis 2) with Bob 90° (basis 1) → should be perfectly correlated

    For BELL TEST, we use combinations for CHSH inequality:
    - Alice bases 0,1 with Bob bases 0,1
    """
    # Key generation: pairs with high expected correlation
    matching = [i for i, (a, b) in enumerate(zip(alice_bases, bob_bases))
                if (a == 1 and b == 0) or (a == 2 and b == 1)]

    # Bell test: all combinations of Alice's first two bases with Bob's first two bases
    bell_test = [i for i, (a, b) in enumerate(zip(alice_bases, bob_bases))
                 if a in [0, 1] and b in [0, 1]]

    return matching, bell_test


In [ ]:
# ============================================================================
# CELL 5: Alice
# ============================================================================
class Alice:
    def __init__(self, noise_model: Optional[NoiseModel] = None):
        self.angles = [0, np.pi/4, np.pi/2]
        self.bases, self.results, self.key = [], [], []
        self.measurer = QuantumMeasurement(noise_model)

    def select_bases(self, n: int) -> List[int]:
        self.bases = np.random.choice(3, size=n, p=[1/3, 1/3, 1/3]).tolist()
        return self.bases

    def measure_qubits(self, pairs: List[EntangledPair], bob_bases: List[int], bob_angles: List[float]) -> List[int]:
        self.results = []
        for i, pair in enumerate(pairs):
            alice_result, _ = self.measurer.measure_entangled_pair(pair, self.angles[self.bases[i]], bob_angles[bob_bases[i]])
            self.results.append(alice_result)
        return self.results

    def generate_key(self, bob_bases: List[int]) -> List[int]:
        self.key = [(self.results[i] + 1) // 2 for i, (a, b) in enumerate(zip(self.bases, bob_bases))
                    if (a == 1 and b == 0) or (a == 2 and b == 1)]
        return self.key

In [ ]:
# ============================================================================
# CELL 6: Eve
# ============================================================================
class Eve:
    """
    Eve performs intercept-resend attack.

    IMPORTANT: This does NOT artificially corrupt quantum states.
    All noise comes from Qiskit's noise model only.

    In a real intercept-resend attack:
    1. Eve intercepts the entangled pair
    2. She measures one qubit in a random basis
    3. She prepares a new pair based on her measurement
    4. The measurement destroys the original entanglement
    5. This naturally introduces ~25% QBER in ideal conditions

    For simplicity, we track which pairs are attacked without
    modifying the quantum states - letting Qiskit handle the physics.
    """
    def __init__(self, noise_model: Optional[NoiseModel] = None):
        self.attack_probability = 0.0
        self.intercepted_indices = []
        self.measurer = QuantumMeasurement(noise_model)

    def set_attack_probability(self, probability: float):
        self.attack_probability = max(0.0, min(1.0, probability))

    def eavesdrop(self, pairs: List[EntangledPair], alice_bases: List[int], bob_bases: List[int]) -> Tuple[List[EntangledPair], Dict]:
        """
        Track which pairs Eve intercepts.
        Returns original pairs unchanged - no artificial corruption.
        """
        attacked_indices = []

        for i in range(len(pairs)):
            if np.random.random() < self.attack_probability:
                attacked_indices.append(i)

        self.intercepted_indices = attacked_indices
        attack_stats = {
            'attacked': len(attacked_indices),
            'attack_rate': len(attacked_indices) / len(pairs) if pairs else 0
        }

        # Return ORIGINAL pairs - NO artificial state corruption
        return pairs, attack_stats


In [ ]:

# ============================================================================
# CELL 7: Bob
# ============================================================================
class Bob:
    def __init__(self, noise_model: Optional[NoiseModel] = None):
        self.angles = [np.pi/4, np.pi/2, 3*np.pi/4]
        self.bases, self.results, self.key = [], [], []
        self.measurer = QuantumMeasurement(noise_model)

    def select_bases(self, n: int) -> List[int]:
        self.bases = np.random.choice(3, size=n, p=[1/3, 1/3, 1/3]).tolist()
        return self.bases

    def measure_qubits(self, pairs: List[EntangledPair], alice_bases: List[int], alice_angles: List[float]) -> List[int]:
        self.results = []
        for i, pair in enumerate(pairs):
            _, bob_result = self.measurer.measure_entangled_pair(pair, alice_angles[alice_bases[i]], self.angles[self.bases[i]])
            self.results.append(bob_result)
        return self.results

    def generate_key(self, alice_bases: List[int]) -> List[int]:
        self.key = [(self.results[i] + 1) // 2 for i, (a, b) in enumerate(zip(alice_bases, self.bases))
                    if (a == 1 and b == 0) or (a == 2 and b == 1)]

In [ ]:
# ============================================================================
# CELL 8: Bell Test
# ============================================================================
def calculate_correlation(alice_results: List[int], bob_results: List[int],
                         alice_bases: List[int], bob_bases: List[int],
                         a_basis: int, b_basis: int) -> Tuple[float, int]:
    """
    Calculate correlation E(a,b) for measurements at specific basis angles.

    For entangled pairs measured at angles a and b:
    E(a,b) = (# same outcomes - # different outcomes) / (total measurements)

    In quantum mechanics: E(a,b) = -cos(2*(a - b)) for Bell states
    Perfect correlation at same angle: E(a,a) = -1 (always opposite for singlet)
    or E(a,a) = +1 (always same for phi_plus)
    """
    same_count = 0
    diff_count = 0
    total = 0

    for a_res, b_res, a_bas, b_bas in zip(alice_results, bob_results, alice_bases, bob_bases):
        if a_bas == a_basis and b_bas == b_basis:
            total += 1
            if a_res == b_res:
                same_count += 1
            else:
                diff_count += 1

    if total == 0:
        return (0.0, 0)

    # Correlation: E = (same - diff) / total
    # This gives values from -1 (perfect anti-correlation) to +1 (perfect correlation)
    correlation = (same_count - diff_count) / total

    return (correlation, total)


def test_chsh_inequality(alice_results: List[int], bob_results: List[int],
                        alice_bases: List[int], bob_bases: List[int]) -> Dict:
    """
    Test CHSH inequality: S = |E(a,b) - E(a,b') + E(a',b) + E(a',b')|

    Classical bound: S ≤ 2
    Quantum bound: S ≤ 2√2 ≈ 2.828

    In E91:
    - a = 0° (Alice basis 0)
    - a' = 45° (Alice basis 1)
    - b = 45° (Bob basis 0)
    - b' = 90° (Bob basis 1)

    So we calculate:
    - E(0°, 45°) = E(Alice 0, Bob 0)
    - E(0°, 90°) = E(Alice 0, Bob 1)
    - E(45°, 45°) = E(Alice 1, Bob 0)
    - E(45°, 90°) = E(Alice 1, Bob 1)
    """
    # Calculate correlations for all 4 basis combinations
    E00, n00 = calculate_correlation(alice_results, bob_results, alice_bases, bob_bases, 0, 0)
    E01, n01 = calculate_correlation(alice_results, bob_results, alice_bases, bob_bases, 0, 1)
    E10, n10 = calculate_correlation(alice_results, bob_results, alice_bases, bob_bases, 1, 0)
    E11, n11 = calculate_correlation(alice_results, bob_results, alice_bases, bob_bases, 1, 1)

    # CHSH parameter: S = |E00 - E01 + E10 + E11|
    # For phi_plus Bell state with proper angles, should get S ≈ 2.828
    S = abs(E00 - E01 + E10 + E11)

    return {
        'S': S,
        'E00': E00, 'E01': E01, 'E10': E10, 'E11': E11,
        'n00': n00, 'n01': n01, 'n10': n10, 'n11': n11,
        'violation': S > 2.0
    }


def print_bell_test_results(results: Dict):
    """Print Bell test results with detailed statistics."""
    status = "✓ QUANTUM VIOLATION" if results['violation'] else "✗ NO VIOLATION"
    print(f"\nBell Test Results:")
    print(f"  S = {results['S']:.3f} (Classical limit: 2.0, Quantum max: 2.828)")
    print(f"  Status: {status}")
    print(f"  Correlations: E00={results['E00']:.3f}, E01={results['E01']:.3f}, E10={results['E10']:.3f}, E11={results['E11']:.3f}")
    print(f"  Sample sizes: n00={results['n00']}, n01={results['n01']}, n10={results['n10']}, n11={results['n11']}")


In [ ]:
# ============================================================================
# CELL 9: Main Protocol
# ============================================================================
def run_e91_protocol(n_pairs: int = 1000,
                     eve_attack: bool = False,
                     eve_probability: float = 0.5,
                     noise_enabled: bool = False,
                     gate_error_1q: float = 0.001,
                     gate_error_2q: float = 0.01,
                     readout_error: float = 0.01,
                     verbose: bool = False) -> Dict:
    """
    Run E91 protocol with corrected noise parameters.

    Args:
        n_pairs: Number of entangled pairs
        eve_attack: Enable eavesdropping
        eve_probability: Probability Eve intercepts a pair
        noise_enabled: Enable quantum noise
        gate_error_1q: 1-qubit gate error rate (default 0.1%)
        gate_error_2q: 2-qubit gate error rate (default 1%)
        readout_error: Measurement error rate (default 1%)
        verbose: Print detailed results
    """
    pairs = generate_entangled_pairs(n_pairs)

    # Create noise model with corrected parameters
    noise_model = None
    if noise_enabled:
        noise_model = create_noise_model(gate_error_1q, gate_error_2q, readout_error)

    alice, bob = Alice(noise_model), Bob(noise_model)
    alice.select_bases(n_pairs)
    bob.select_bases(n_pairs)

    attack_stats = None
    if eve_attack:
        eve = Eve(noise_model)
        eve.set_attack_probability(eve_probability)
        pairs, attack_stats = eve.eavesdrop(pairs, alice.bases, bob.bases)

    alice.measure_qubits(pairs, bob.bases, bob.angles)
    bob.measure_qubits(pairs, alice.bases, alice.angles)

    matching, _ = exchange_basis_information(alice.bases, bob.bases)
    bell_results = test_chsh_inequality(alice.results, bob.results, alice.bases, bob.bases)

    if bell_results['violation']:
        alice.generate_key(bob.bases)
        bob.generate_key(alice.bases)
        qber = sum(1 for a, b in zip(alice.key, bob.key) if a != b) / len(alice.key) if alice.key else 1.0
    else:
        alice.key, bob.key, qber = [], [], 1.0

    success = bell_results['violation'] and len(alice.key) > 0 and qber <= 0.11

    if verbose:
        status = "✓" if success else "✗"
        print(f"E91: Pairs={n_pairs} | S={bell_results['S']:.3f} | Key={len(alice.key)}bits | QBER={qber*100:.1f}% | {status}")

    return {
        'success': success, 'n_pairs': n_pairs, 'n_matching': len(matching),
        'bell_S': bell_results['S'], 'bell_violation': bell_results['violation'],
        'alice_key_length': len(alice.key), 'qber': qber, 'bell_results': bell_results,
        'eve_attack': eve_attack, 'attack_stats': attack_stats
    }

In [ ]:
# ============================================================================
# CELL 10: Noise Analysis
# ============================================================================
def run_noise_analysis(n_pairs: int = 500, noise_levels: List[float] = None, eve_probability: float = 0.0) -> Dict:
    """Uses corrected noise parameters."""
    if noise_levels is None:
        noise_levels = [0.0, 0.01, 0.02, 0.03, 0.05, 0.1]

    metrics = {'noise_levels': [], 'chsh_s': [], 'qber': [], 'key_length': []}

    for noise_prob in noise_levels:
        # Use noise_prob for both 1q and 2q gates (simplified)
        result = run_e91_protocol(
            n_pairs=n_pairs,
            eve_attack=(eve_probability > 0),
            eve_probability=eve_probability,
            noise_enabled=(noise_prob > 0),
            gate_error_1q=noise_prob,
            gate_error_2q=noise_prob * 2,  # 2q gates typically 2x worse
            readout_error=noise_prob,
            verbose=False
        )
        metrics['noise_levels'].append(noise_prob)
        metrics['chsh_s'].append(result['bell_S'])
        metrics['qber'].append(result['qber'])
        metrics['key_length'].append(result['alice_key_length'])

    return metrics


def plot_noise_impact(metrics: Dict, title_suffix: str = ""):
    """Create 3-panel plot showing key E91 metrics vs noise."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'E91 Protocol: Key Metrics vs Noise{title_suffix}',
                 fontsize=14, fontweight='bold')

    noise_percent = np.array(metrics['noise_levels']) * 100

    # Plot 1: CHSH Parameter
    ax1 = axes[0]
    ax1.plot(noise_percent, metrics['chsh_s'], 'b-o', linewidth=2.5, markersize=9)
    ax1.axhline(y=2.0, color='red', linestyle='--', linewidth=2, label='Classical bound')
    ax1.axhline(y=2*np.sqrt(2), color='green', linestyle='--', linewidth=2, alpha=0.5, label='Quantum max')
    ax1.fill_between(noise_percent, 2.0, 2*np.sqrt(2), alpha=0.15, color='green')
    ax1.set_xlabel('Noise Level (%)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('CHSH Parameter (S)', fontsize=12, fontweight='bold')
    ax1.set_title('Bell Violation', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([1.5, 3.0])

    # Plot 2: QBER
    ax2 = axes[1]
    qber_percent = np.array(metrics['qber']) * 100
    ax2.plot(noise_percent, qber_percent, 'r-o', linewidth=2.5, markersize=9)
    ax2.axhline(y=11, color='orange', linestyle='--', linewidth=2, label='Security limit')
    ax2.fill_between(noise_percent, 0, 11, alpha=0.15, color='green', label='Safe zone')
    ax2.set_xlabel('Noise Level (%)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('QBER (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Error Rate', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    # Plot 3: Key Length
    ax3 = axes[2]
    ax3.plot(noise_percent, metrics['key_length'], 'g-o', linewidth=2.5, markersize=9)
    ax3.set_xlabel('Noise Level (%)', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Key Length (bits)', fontsize=12, fontweight='bold')
    ax3.set_title('Key Generation', fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def compare_scenarios(n_pairs: int = 500):
    """Run complete comparison analysis."""
    print("="*60 + "\nE91 NOISE ANALYSIS\n" + "="*60)

    metrics_clean = run_noise_analysis(n_pairs=n_pairs, eve_probability=0.0)
    metrics_eve = run_noise_analysis(n_pairs=n_pairs, eve_probability=0.3)

    plot_noise_impact(metrics_clean, title_suffix=" (Clean)")
    plot_noise_impact(metrics_eve, title_suffix=" (Eve 30%)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Clean vs Eve Attack', fontsize=14, fontweight='bold')
    noise_percent = np.array(metrics_clean['noise_levels']) * 100

    for ax, data_c, data_e, ylabel, title in [
        (axes[0], metrics_clean['chsh_s'], metrics_eve['chsh_s'], 'S', 'Bell Violation'),
        (axes[1], np.array(metrics_clean['qber'])*100, np.array(metrics_eve['qber'])*100, 'QBER%', 'Error Rate'),
        (axes[2], metrics_clean['key_length'], metrics_eve['key_length'], 'Key', 'Key Length')
    ]:
        ax.plot(noise_percent, data_c, 'g-o', linewidth=2.5, markersize=8, label='Clean')
        ax.plot(noise_percent, data_e, 'r-s', linewidth=2.5, markersize=8, label='Eve')
        ax.set_xlabel('Noise%', fontweight='bold')
        ax.set_ylabel(ylabel, fontweight='bold')
        ax.set_title(title, fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\n{'Noise%':<8} {'Type':<8} {'S':<8} {'QBER%':<8} {'Key'}")
    print("-"*50)
    for i, n in enumerate(metrics_clean['noise_levels']):
        print(f"{n*100:>5.1f}    Clean    {metrics_clean['chsh_s'][i]:.3f}    {metrics_clean['qber'][i]*100:>5.1f}    {metrics_clean['key_length'][i]}")
        print(f"         Eve30%   {metrics_eve['chsh_s'][i]:.3f}    {metrics_eve['qber'][i]*100:>5.1f}    {metrics_eve['key_length'][i]}")
    print("="*60)

    return metrics_clean, metrics_eve

In [ ]:
# ============================================================================
# CELL 11: Run Analysis
# ============================================================================

print("="*70)
print("E91 QUANTUM KEY DISTRIBUTION PROTOCOL - COMPREHENSIVE TESTS")
print("="*70)

# Test 1: Ideal conditions (no noise, no Eve)
print("\n" + "="*70)
print("TEST 1: IDEAL CONDITIONS (No noise, No Eve)")
print("="*70)
print("Expected: S ≈ 2.7-2.8, QBER < 5%, Key length ≈ 33% of pairs")
result = run_e91_protocol(n_pairs=500, noise_enabled=False, eve_attack=False, verbose=True)
print(f"\nDetailed Results:")
print(f"  Bell S Parameter: {result['bell_S']:.4f}")
print(f"  Bell Violation: {'YES ✓' if result['bell_violation'] else 'NO ✗'}")
print(f"  Key Length: {result['alice_key_length']} bits from {result['n_pairs']} pairs")
print(f"  QBER: {result['qber']*100:.2f}%")
print(f"  Success: {'YES ✓' if result['success'] else 'NO ✗'}")

# Test 2: With realistic noise
print("\n" + "="*70)
print("TEST 2: REALISTIC QUANTUM HARDWARE NOISE")
print("="*70)
print("Gate errors: 1-qubit=0.1%, 2-qubit=1%, Readout=1%")
print("Expected: S ≈ 2.5-2.7, QBER 1-3%, Key length ≈ 30% of pairs")
result = run_e91_protocol(n_pairs=500, noise_enabled=True,
                         gate_error_1q=0.001, gate_error_2q=0.01,
                         readout_error=0.01, eve_attack=False, verbose=True)
print(f"\nDetailed Results:")
print(f"  Bell S Parameter: {result['bell_S']:.4f}")
print(f"  Bell Violation: {'YES ✓' if result['bell_violation'] else 'NO ✗'}")
print(f"  Key Length: {result['alice_key_length']} bits")
print(f"  QBER: {result['qber']*100:.2f}%")
print(f"  Success: {'YES ✓' if result['success'] else 'NO ✗'}")

# Test 3: Eve attack (20%)
print("\n" + "="*70)
print("TEST 3: MILD EVE ATTACK (20% interception)")
print("="*70)
print("Expected: Increased QBER, reduced S parameter")
result = run_e91_protocol(n_pairs=500, noise_enabled=False,
                         eve_attack=True, eve_probability=0.2, verbose=True)
print(f"\nDetailed Results:")
print(f"  Bell S Parameter: {result['bell_S']:.4f}")
print(f"  Bell Violation: {'YES ✓' if result['bell_violation'] else 'NO ✗'}")
print(f"  Key Length: {result['alice_key_length']} bits")
print(f"  QBER: {result['qber']*100:.2f}%")
print(f"  Eve attacked: {result['attack_stats']['attacked']} pairs ({result['attack_stats']['attack_rate']*100:.1f}%)")
print(f"  Success: {'YES ✓' if result['success'] else 'NO ✗'}")

# Test 4: Eve attack (50%)
print("\n" + "="*70)
print("TEST 4: STRONG EVE ATTACK (50% interception)")
print("="*70)
print("Expected: QBER > 11%, protocol should FAIL security check")
result = run_e91_protocol(n_pairs=500, noise_enabled=False,
                         eve_attack=True, eve_probability=0.5, verbose=True)
print(f"\nDetailed Results:")
print(f"  Bell S Parameter: {result['bell_S']:.4f}")
print(f"  Bell Violation: {'YES ✓' if result['bell_violation'] else 'NO ✗'}")
print(f"  Key Length: {result['alice_key_length']} bits")
print(f"  QBER: {result['qber']*100:.2f}%")
print(f"  Eve attacked: {result['attack_stats']['attacked']} pairs ({result['attack_stats']['attack_rate']*100:.1f}%)")
print(f"  Success: {'YES ✓' if result['success'] else 'NO ✗'}")

# Test 5: Combined noise + Eve
print("\n" + "="*70)
print("TEST 5: REALISTIC SCENARIO (Noise + Mild Eve Attack)")
print("="*70)
print("Gate errors: 1%, Eve interception: 15%")
result = run_e91_protocol(n_pairs=500, noise_enabled=True,
                         gate_error_1q=0.01, gate_error_2q=0.02,
                         readout_error=0.01,
                         eve_attack=True, eve_probability=0.15, verbose=True)
print(f"\nDetailed Results:")
print(f"  Bell S Parameter: {result['bell_S']:.4f}")
print(f"  Bell Violation: {'YES ✓' if result['bell_violation'] else 'NO ✗'}")
print(f"  Key Length: {result['alice_key_length']} bits")
print(f"  QBER: {result['qber']*100:.2f}%")
print(f"  Eve attacked: {result['attack_stats']['attacked']} pairs")
print(f"  Success: {'YES ✓' if result['success'] else 'NO ✗'}")

print("\n" + "="*70)
print("TESTS COMPLETE")
print("="*70)

# Now run the full noise analysis
print("\n\n" + "="*70)
print("COMPREHENSIVE NOISE ANALYSIS")
print("="*70)
metrics_clean, metrics_attacked = compare_scenarios(n_pairs=500)
